<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_2_9_LogReg_OI_C9_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mastering Logistic Regression: Theory & Case Studies

**Attribution:** Exercises adapted from *Introduction to Modern Statistics (2e)* Chapter 9.6.  
**Source:** [OpenIntro IMS](https://openintro-ims.netlify.app/model-logistic)  
**License:** [CC BY-SA 3.0 US](https://creativecommons.org/licenses/by-sa/3.0/us/)

---

## Introduction
While previous notebooks focused on the **Machine Learning workflow** (pipelines, accuracy, ROC curves), this notebook explores the **Statistical foundations** of logistic regression. We will look at why we use the logistic curve instead of a line, how to interpret coefficients as "Odds Ratios," and analyze safety-critical data from the Challenger space shuttle disaster.

### Learning Objectives
1. Understand the mathematical constraints of the logistic function.
2. Interpret coefficients as **Odds Ratios**.
3. Identify and handle **Multicollinearity**.
4. Evaluate model fit using **AIC** (Akaike Information Criterion).
5. Apply models to safety-critical decisions (The Challenger Disaster).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import os

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (12, 6)

## Part 1: Why Not Linear Regression?

### Exercise 1.1: Fact Checking
Determine which of the following statements are **True** or **False**. For each false statement, explain why.

1. In logistic regression, we fit a line to model the relationship between the predictors and the binary outcome.
2. In logistic regression, we expect the residuals to be evenly scattered on either side of zero, just like with linear regression.
3. In logistic regression, the outcome variable is binary, but the predictor variable(s) can be either binary or continuous.

### Visual Intuition
Run the cell below to see the danger of using a straight line for binary data. Note how the red line predicts probabilities **above 1.0** and **below 0.0**—which are mathematically impossible. The green S-curve (the sigmoid) is naturally constrained between 0 and 1.

In [ ]:
np.random.seed(42)
x = np.linspace(-5, 5, 40)
# Generate binary data with a probability that follows a sigmoid curve
y_prob = 1 / (1 + np.exp(-x))
y = (y_prob > np.random.rand(40)).astype(int)

fig, ax = plt.subplots()
sns.regplot(x=x, y=y, logistic=False, ci=None, line_kws={'color':'red', 'label':'Linear Fit (OLS)'}, ax=ax)
sns.regplot(x=x, y=y, logistic=True, ci=None, line_kws={'color':'green', 'label':'Logistic Fit'}, ax=ax)

ax.set_title("Linear vs. Logistic Fit on Binary Data")
ax.set_ylabel("Probability / Outcome")
ax.axhline(0, color='black', linestyle='--', alpha=0.3)
ax.axhline(1, color='black', linestyle='--', alpha=0.3)
ax.legend()
plt.show()

--- 
## Part 2: Multicollinearity in Possums

We have data from 104 brushtail possums. We want to predict whether a possum is from the Victoria population (`pop=Vic`) or another region (`pop=other`) based on physical measurements.

**Multicollinearity** occurs when two or more predictors are highly correlated. This can "confuse" the model, making it hard to tell which variable is actually responsible for the effect. 

In [ ]:
# Robust data loading for Colab and local environments
url = "https://www.openintro.org/data/csv/possum.csv"
if not os.path.exists('possum.csv'):
    import urllib.request
    # We use a custom User-Agent to avoid blocks
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(url, 'possum.csv')

possum = pd.read_csv('possum.csv')
possum['vic_pop'] = (possum['pop'] == 'Vic').astype(int)
possum['sex_binary'] = (possum['sex'] == 'm').astype(int)
possum.head()

### Student Task: Correlation & Modeling
1. Inspect the correlation between `head_l` (head length) and `skull_w` (skull width).
2. Fit two models using `statsmodels` (which provides detailed summary tables and AIC):
    *   **Full Model:** `vic_pop ~ head_l + skull_w + total_l + tail_l` 
    *   **Reduced Model:** `vic_pop ~ skull_w + total_l + tail_l` (removing head length)
3. Look at the **P-values** and **AIC** for both. Does the significance of `skull_w` change when `head_l` is removed?

In [ ]:
# 1. Correlation Heatmap
measures = ['head_l', 'skull_w', 'total_l', 'tail_l']
sns.heatmap(possum[measures].corr(), annot=True, cmap='RdBu_r', center=0)
plt.title("Predictor Correlations")
plt.show()

# 2. Modeling with statsmodels (smf)
# We use statsmodels here because it provides the 'AIC' and P-values used in IMS Chapter 9.
full_model = smf.logit('vic_pop ~ sex_binary + head_l + skull_w + total_l + tail_l', data=possum).fit(disp=0)
reduced_model = smf.logit('vic_pop ~ sex_binary + skull_w + total_l + tail_l', data=possum).fit(disp=0)

print("--- FULL MODEL SUMMARY ---")
print(full_model.summary().tables[1])
print(f"Full Model AIC: {full_model.aic:.2f}")

print("\n--- REDUCED MODEL SUMMARY ---")
print(reduced_model.summary().tables[1])
print(f"Reduced Model AIC: {reduced_model.aic:.2f}")

--- 
## Part 3: The Challenger Disaster

On January 28, 1986, the Space Shuttle Challenger exploded 73 seconds into flight. The failure was caused by O-ring seals that lost their elasticity in cold temperatures. 

Engineers had warned that launching at 31°F was dangerous, but the data they presented was confusing. They only showed flights that had *already failed*, rather than looking at the relationship between temperature and failure across *all* flights.

The complete flight history leads to the following logistic regression model ($p$ = probability of damage):

$$\text{logit}(p) = 11.6630 - 0.2162 \times \text{temperature}$$

### Student Task: Risk Assessment
1. Using the formula, calculate the **log-odds** and then the **probability** of O-ring damage at 31°F.
2. Visualize the probability curve. 
3. **Discussion:** Look at the plot. Why was it so hard for decision-makers to see the trend with the original flight data? (Hint: Most previous flights were in weather warmer than 60°F).

In [ ]:
def logit_to_prob(logit):
    return np.exp(logit) / (1 + np.exp(logit))

temp_launch = 31
log_odds = 11.6630 - 0.2162 * temp_launch
prob_failure = logit_to_prob(log_odds)

print(f"Log-odds of damage at 31°F: {log_odds:.4f}")
print(f"Probability of damage at 31°F: {prob_failure:.1%}")

# Visualization
temps = np.linspace(25, 85, 100)
probs = logit_to_prob(11.6630 - 0.2162 * temps)

plt.plot(temps, probs, color='red', lw=3, label='Model Prediction')
plt.fill_between(temps, 0, probs, color='red', alpha=0.1)
plt.axvline(31, color='black', linestyle='--', label='Launch Day (31°F)')
plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='50/50 Risk')

plt.xlabel("Temperature (°F)")
plt.ylabel("Predicted Probability of O-ring Damage")
plt.title("Challenger Risk Analysis: The Sigmoid Curve")
plt.legend()
plt.show()

--- 
## Part 4: The Odds Ratio

In logistic regression, a 1-unit increase in $X$ does not increase $Y$ by a fixed amount. Instead, it increases the **odds** of the outcome by a fixed multiple: the **Odds Ratio (OR)**.

$$\text{OR} = e^{\beta_1}$$

### Exercise: Interpretation
Suppose you are building a spam filter. The coefficient ($\beta$) for the word "Winner" is **1.63**.
1. Calculate the Odds Ratio.
2. Complete this sentence: "An email containing the word 'Winner' is ______ times more likely to be spam than an email without it, holding all other factors constant."

## Solutions Appendix (Self-Check)

### Part 1: Why Not Linear?
1. **False.** We fit a logistic (S-shaped) curve. Linear regression fits a line, but it can predict probabilities less than 0 or greater than 1.
2. **False.** Residuals in logistic regression are not expected to be normal or evenly scattered because the outcome is constrained to 0 and 1.
3. **True.** Predictors can be any type; only the outcome must be binary.

### Part 2: Multicollinearity
When `head_l` is removed, the coefficient for `skull_w` becomes much more significant (lower P-value). This is because head length and skull width are highly correlated (0.71). They were "competing" to explain the same variance.

### Part 3: Challenger
The probability of failure at 31°F is **99.3%**. The disaster was statistically predictable, but the raw data was mostly collected at higher temperatures where the risk was near zero, making the trend hard to see without a proper model.

### Part 4: Odds Ratio
The Odds Ratio is $e^{1.63} \approx 5.10$. An email with the word 'Winner' is **5.1 times more likely** to be spam.